<a href="https://colab.research.google.com/github/ladams1204/SportsBookEdge/blob/main/03_features.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
!pip install nba_api --quiet

import sqlite3
import pandas as pd
import numpy as np
from google.colab import files

In [8]:
from nba_api.stats.endpoints import playergamelog
from nba_api.stats.static import players
import time

conn = sqlite3.connect('sportsbook.db')
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS nba_player_games (
    game_id TEXT,
    player_id INTEGER,
    player_name TEXT,
    game_date TEXT,
    matchup TEXT,
    wl TEXT,
    minutes INTEGER,
    points INTEGER,
    rebounds INTEGER,
    assists INTEGER,
    steals INTEGER,
    blocks INTEGER,
    turnovers INTEGER,
    fgm INTEGER,
    fga INTEGER,
    fg3m INTEGER,
    fg3a INTEGER,
    plus_minus INTEGER,
    season TEXT,
    PRIMARY KEY (game_id, player_id)
)
''')
conn.commit()

target_players = [
    "LeBron James", "Stephen Curry", "Nikola Jokic", "Luka Doncic",
    "Jayson Tatum", "Giannis Antetokounmpo", "Shai Gilgeous-Alexander",
    "Kevin Durant", "Anthony Edwards", "Joel Embiid",
    "Devin Booker", "Damian Lillard", "Jimmy Butler", "Paul George",
    "Donovan Mitchell", "Trae Young", "De'Aaron Fox", "Ja Morant",
    "Anthony Davis", "Victor Wembanyama"
]

def pull_player_season(player_name, season):
    matches = players.find_players_by_full_name(player_name)
    if not matches:
        return 0
    player = matches[0]
    log = playergamelog.PlayerGameLog(player_id=player['id'], season=season)
    df = log.get_data_frames()[0]
    if len(df) == 0:
        return 0
    df['SEASON'] = season
    df['PLAYER_NAME'] = player['full_name']
    df = df.rename(columns={
        'Game_ID': 'game_id', 'Player_ID': 'player_id',
        'PLAYER_NAME': 'player_name', 'GAME_DATE': 'game_date',
        'MATCHUP': 'matchup', 'WL': 'wl', 'MIN': 'minutes',
        'PTS': 'points', 'REB': 'rebounds', 'AST': 'assists',
        'STL': 'steals', 'BLK': 'blocks', 'TOV': 'turnovers',
        'FGM': 'fgm', 'FGA': 'fga', 'FG3M': 'fg3m', 'FG3A': 'fg3a',
        'PLUS_MINUS': 'plus_minus', 'SEASON': 'season'
    })[['game_id', 'player_id', 'player_name', 'game_date', 'matchup', 'wl',
        'minutes', 'points', 'rebounds', 'assists', 'steals', 'blocks',
        'turnovers', 'fgm', 'fga', 'fg3m', 'fg3a', 'plus_minus', 'season']]
    inserted = 0
    for _, row in df.iterrows():
        try:
            cursor.execute(f"INSERT INTO nba_player_games VALUES ({','.join(['?']*19)})", tuple(row))
            inserted += 1
        except sqlite3.IntegrityError:
            pass
    conn.commit()
    return inserted

total = 0
for season in ['2024-25', '2025-26']:
    for name in target_players:
        n = pull_player_season(name, season)
        total += n
        print(f"{season} {name}: {n} games")
        time.sleep(0.6)

print(f"\nTotal inserted: {total}")
count = cursor.execute("SELECT COUNT(*) FROM nba_player_games").fetchone()[0]
print(f"Database rows: {count}")

2024-25 LeBron James: 0 games
2024-25 Stephen Curry: 0 games
2024-25 Nikola Jokic: 0 games
2024-25 Luka Doncic: 0 games
2024-25 Jayson Tatum: 0 games
2024-25 Giannis Antetokounmpo: 0 games
2024-25 Shai Gilgeous-Alexander: 0 games
2024-25 Kevin Durant: 0 games
2024-25 Anthony Edwards: 0 games
2024-25 Joel Embiid: 0 games
2024-25 Devin Booker: 0 games
2024-25 Damian Lillard: 0 games
2024-25 Jimmy Butler: 0 games
2024-25 Paul George: 0 games
2024-25 Donovan Mitchell: 0 games
2024-25 Trae Young: 0 games
2024-25 De'Aaron Fox: 0 games
2024-25 Ja Morant: 0 games
2024-25 Anthony Davis: 0 games
2024-25 Victor Wembanyama: 0 games
2025-26 LeBron James: 0 games
2025-26 Stephen Curry: 0 games
2025-26 Nikola Jokic: 0 games
2025-26 Luka Doncic: 0 games
2025-26 Jayson Tatum: 0 games
2025-26 Giannis Antetokounmpo: 0 games
2025-26 Shai Gilgeous-Alexander: 0 games
2025-26 Kevin Durant: 0 games
2025-26 Anthony Edwards: 0 games
2025-26 Joel Embiid: 0 games
2025-26 Devin Booker: 0 games
2025-26 Damian Lilla

In [9]:
df = pd.read_sql("""
    SELECT * FROM nba_player_games
    ORDER BY player_id, game_date
""", conn)

df['game_date'] = pd.to_datetime(df['game_date'])

numeric_cols = ['minutes', 'points', 'rebounds', 'assists', 'steals', 'blocks',
                'turnovers', 'fgm', 'fga', 'fg3m', 'fg3a', 'plus_minus']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(f"Loaded {len(df)} rows")
print(f"Columns: {df.columns.tolist()}")
print(f"\nDate range: {df['game_date'].min().date()} to {df['game_date'].max().date()}")
print(f"Players: {df['player_name'].nunique()}")
df.head()

Loaded 2149 rows
Columns: ['game_id', 'player_id', 'player_name', 'game_date', 'matchup', 'wl', 'minutes', 'points', 'rebounds', 'assists', 'steals', 'blocks', 'turnovers', 'fgm', 'fga', 'fg3m', 'fg3a', 'plus_minus', 'season']

Date range: 2024-10-22 to 2026-04-12
Players: 20


,game_id,player_id,player_name,game_date,matchup,wl,minutes,points,rebounds,assists,steals,blocks,turnovers,fgm,fga,fg3m,fg3a,plus_minus,season
0,0022501116,2544,LeBron James,2026-04-02,LAL @ OKC,L,26,13,6,2,1,1,2,3,7,1,4,-37,2025-26
1,0022401117,2544,LeBron James,2025-04-03,LAL vs. GSW,L,40,33,5,9,1,1,4,10,15,5,8,-1,2024-25
2,0022401126,2544,LeBron James,2025-04-04,LAL vs. NOP,W,33,27,0,8,2,0,1,9,17,2,7,7,2024-25
3,0022501140,2544,LeBron James,2026-04-05,LAL @ DAL,L,39,30,9,15,1,0,4,12,22,1,6,1,2025-26
4,0022401135,2544,LeBron James,2025-04-06,LAL @ OKC,W,34,19,3,7,1,0,2,9,16,1,1,7,2024-25


In [10]:
df = df.sort_values(['player_id', 'game_date']).reset_index(drop=True)

df['pts_avg_10'] = (
    df.groupby('player_id')['points']
      .transform(lambda x: x.shift(1).rolling(10, min_periods=3).mean())
)

lebron_check = df[df['player_name'] == 'LeBron James'][
    ['game_date', 'matchup', 'points', 'pts_avg_10']
].tail(15)
print(lebron_check.to_string(index=False))

 game_date     matchup  points  pts_avg_10
2026-03-14 LAL vs. DEN      17        19.8
2026-03-16   LAL @ HOU      18        18.7
2026-03-18   LAL @ HOU      30        19.2
2026-03-19   LAL @ MIA      19        20.2
2026-03-21   LAL @ ORL      12        20.0
2026-03-23   LAL @ DET      12        19.7
2026-03-25   LAL @ IND      23        18.7
2026-03-27 LAL vs. BKN      14        18.6
2026-03-30 LAL vs. WAS      21        17.9
2026-03-31 LAL vs. CLE      14        18.4
2026-04-02   LAL @ OKC      13        18.0
2026-04-05   LAL @ DAL      30        17.6
2026-04-09   LAL @ GSW      26        18.8
2026-04-10 LAL vs. PHX      28        18.4
2026-04-12 LAL vs. UTA      18        19.3


In [11]:
# Feature 2: Rolling 10-game rebounds average
df['reb_avg_10'] = (
    df.groupby('player_id')['rebounds']
      .transform(lambda x: x.shift(1).rolling(10, min_periods=3).mean())
)

# Feature 3: Rolling 10-game assists average
df['ast_avg_10'] = (
    df.groupby('player_id')['assists']
      .transform(lambda x: x.shift(1).rolling(10, min_periods=3).mean())
)

# Feature 4: Rolling 10-game minutes average (important — a player's points depend
# heavily on how many minutes they're playing; injuries/benchings show up here first)
df['min_avg_10'] = (
    df.groupby('player_id')['minutes']
      .transform(lambda x: x.shift(1).rolling(10, min_periods=3).mean())
)

# Feature 5: Rest days since last game
# We compute days between consecutive games for each player
df['rest_days'] = (
    df.groupby('player_id')['game_date']
      .transform(lambda x: (x - x.shift(1)).dt.days)
)

# Feature 6: Back-to-back flag (1 if this game is day after previous game, else 0)
df['is_b2b'] = (df['rest_days'] == 1).astype(int)

# Feature 7: Home or away — extract from the matchup string
# Matchups look like "LAL vs. UTA" (home) or "LAL @ OKC" (away)
df['is_home'] = df['matchup'].str.contains('vs').astype(int)

# Feature 8: Rolling 20-game points average (longer window = more stable signal)
df['pts_avg_20'] = (
    df.groupby('player_id')['points']
      .transform(lambda x: x.shift(1).rolling(20, min_periods=5).mean())
)

# Verify: show LeBron's last 10 rows with ALL the new features
cols_to_show = ['game_date', 'matchup', 'points', 'pts_avg_10', 'pts_avg_20',
                'reb_avg_10', 'ast_avg_10', 'min_avg_10', 'rest_days', 'is_b2b', 'is_home']
lebron_features = df[df['player_name'] == 'LeBron James'][cols_to_show].tail(10)
print(lebron_features.to_string(index=False))

 game_date     matchup  points  pts_avg_10  pts_avg_20  reb_avg_10  ast_avg_10  min_avg_10  rest_days  is_b2b  is_home
2026-03-23   LAL @ DET      12        19.7       20.00         6.4         6.2        33.4        2.0       0        0
2026-03-25   LAL @ IND      23        18.7       19.50         6.6         6.3        34.5        2.0       0        0
2026-03-27 LAL vs. BKN      14        18.6       19.40         7.4         6.7        35.2        2.0       0        1
2026-03-30 LAL vs. WAS      21        17.9       19.25         7.3         6.8        35.6        3.0       0        1
2026-03-31 LAL vs. CLE      14        18.4       19.30         7.8         7.2        35.5        1.0       1        1
2026-04-02   LAL @ OKC      13        18.0       18.90         7.6         7.1        35.3        2.0       0        0
2026-04-05   LAL @ DAL      30        17.6       18.15         7.6         6.8        33.9        3.0       0        0
2026-04-09   LAL @ GSW      26        18.8      

In [12]:
# Save the DataFrame (with features) back to the database as a new table
df.to_sql('nba_features', conn, if_exists='replace', index=False)
conn.commit()

# Verify
tables = cursor.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall()
print("Tables in database:", tables)

count = cursor.execute("SELECT COUNT(*) FROM nba_features").fetchone()[0]
print(f"nba_features table: {count} rows")

# Show what columns we ended up with
print(f"\nFeatures available: {df.columns.tolist()}")

Tables in database: [('nba_player_games',), ('nba_features',)]
nba_features table: 2149 rows

Features available: ['game_id', 'player_id', 'player_name', 'game_date', 'matchup', 'wl', 'minutes', 'points', 'rebounds', 'assists', 'steals', 'blocks', 'turnovers', 'fgm', 'fga', 'fg3m', 'fg3a', 'plus_minus', 'season', 'pts_avg_10', 'reb_avg_10', 'ast_avg_10', 'min_avg_10', 'rest_days', 'is_b2b', 'is_home', 'pts_avg_20']
